# Lock-in Equilibrium Propagation for Phasor Networks

This notebook derives and demonstrates the **lock-in EP** method
(`LockinEP` in `src/ep.jl`) — the temporal analog of holomorphic EP's
spatial Cauchy contour. It is the third notebook in the EP series:

- `ep_demo.ipynb` — vanilla EP on real-weight networks; small-β dilemma.
- `hep_demo.ipynb` — holomorphic EP; *spatial* contour `β = r·e^{2πin/N}`
  on the complex β-plane. Requires a holomorphic activation.
- `phasor_ep_demo.ipynb` — phasor EP integration with the package API.
- **This notebook** — lock-in EP: sweep `β(t) = ε·cos(ω_p·t)` through
  time and demodulate at `+ω_p` to recover the gradient. Works on the
  phasor recurrence, where spatial hEP fails.

We will:

1. Show why hEP's spatial contour cannot be reused on the phasor
   recurrence (the unit-magnitude projector is non-holomorphic).
2. Derive the temporal lock-in coefficient and the choice of real
   vs. complex probe.
3. Walk through a single gradient extraction step by step.
4. Map the three accuracy knobs `(ε, ω_p, n_cycles)` and their failure
   modes.
5. Train an XOR classifier and compare to static EP head-to-head.

The full formal write-up lives in `docs/phasor_ep_design.md` §
"Temporal Cauchy / lock-in detection for phasor EP".

In [ ]:
using Pkg; Pkg.activate("..")
using LinearAlgebra, Statistics, Random, Plots
using Random: Xoshiro
using Lux, PhasorNetworks

## 1. Why spatial hEP doesn't transfer to phasor networks

Spatial hEP recovers the loss gradient via Cauchy's integral formula on
a circle in the complex $\beta$ plane,

$$\frac{\partial \mathcal{L}}{\partial \beta} \approx
  \frac{1}{N} \sum_{n=1}^{N} \frac{\mathcal{L}(\beta_n)}{\beta_n},
  \qquad \beta_n = r \cdot e^{2\pi i n / N}.$$

This works when the network's equilibrium $z^\star(\beta)$ is
**holomorphic** in $\beta$. Phasor EP's settling step uses the
unit-modulus projector

$$\text{unit\_project}(z) = \frac{z}{|z|}, \qquad
  |z| = \sqrt{z \cdot \bar z},$$

which involves the conjugate $\bar z$. The equilibrium therefore
depends on both $\beta$ and $\bar\beta$:

$$z^\star(\beta) = z^\star(\beta, \bar\beta).$$

Substituting $\beta = r e^{i\varphi}$ into the contour formula picks
up a contaminant proportional to $\bar\beta / \beta^2 \propto
e^{-2i\varphi} / r^2$ — a conjugate sideband that **does not decrease
with $N$** and does not decrease with $r$ either. Spatial hEP is
fundamentally incompatible with non-holomorphic recurrences. We need
a different way to probe the linear response.

## 2. Temporal Cauchy as the cure

Instead of sampling $\beta$ on a *spatial* contour in $\mathbb{C}$,
sweep it through time as a real (or complex) probe and read the
network's response in the frequency domain.

**Real probe** (the one `LockinEP` uses):

$$\beta(t) = \varepsilon \cos(\omega_p t).$$

For small $\varepsilon$, the equilibrium follows the probe to first
order in $\varepsilon$:

$$z^\star(t) = z^\star(0)
  + \varepsilon e^{i\omega_p t} \frac{\partial z^\star}{\partial \beta}
  + \varepsilon e^{-i\omega_p t} \frac{\partial z^\star}{\partial \bar\beta}
  + O(\varepsilon^2).$$

**Lock-in detection** multiplies any observable $f(t)$ by
$e^{-i\omega_p t}$ and time-averages over an integer number of probe
cycles:

$$H_+ = \frac{1}{T_{\text{int}}} \int_0^{T_{\text{int}}}
  f(t) \cdot e^{-i\omega_p t} \, dt.$$

The two probe choices extract different things:

| probe | $2 \cdot \mathrm{Re}(H_+/\varepsilon)$ extracts | matches FD on real $W$? |
|---|---|---|
| real $\cos(\omega_p t)$ | $\partial z^\star / \partial \beta + \partial z^\star / \partial \bar\beta = d/d\beta_\text{real}$ | **yes** |
| complex $e^{i\omega_p t}$ | $\partial z^\star / \partial \beta$ (Wirtinger only) | only if holomorphic in $\beta$ |

When the recurrence is holomorphic in $\beta$ (hEP regime),
$\partial/\partial\bar\beta = 0$ and the two probes agree. When it is
not (the phasor-EP regime), only the **real probe** reproduces FD on
real-valued weights. This is why `LockinEP` uses $\cos(\omega_p t)$,
not $e^{i\omega_p t}$.

In [ ]:
# The two probe trajectories on the complex β-plane.
# Real probe stays on the real axis; complex probe traces a circle.
ε   = 0.5
ω_p = 0.05
ts  = 0:1.0:200

β_real    = [ε * cos(ω_p * t) + 0im for t in ts]
β_complex = [ε * exp(im * ω_p * t)  for t in ts]

p = plot(real.(β_real), imag.(β_real), lw=2, label="real probe ε·cos(ω_p t)",
         color=:steelblue, aspect_ratio=:equal)
plot!(real.(β_complex), imag.(β_complex), lw=2, label="complex probe ε·e^{iω_p t}",
      color=:darkorange)
scatter!([0], [0], color=:black, label="β = 0", markersize=4)
plot!(xlabel="Re(β)", ylabel="Im(β)",
      title="Two probe trajectories in the β-plane",
      legend=:topright)

## 3. Single-equilibrium efficiency

Static EP needs **two equilibria** per gradient:

```
free phase  (β = 0)   →  settle T_free  steps   →  z*_free
nudged phase (β > 0)  →  settle T_nudge steps   →  z*_nudge

grad = -(hebb(z*_nudge) - hebb(z*_free)) / β
```

Lock-in EP needs **one** equilibrium plus one driven trajectory:

```
free phase  (β = 0)        →  settle T_free steps      →  z*_free
warm-up     (β(t) = ε·cos) →  drive T_warmup steps     →  (transient)
lock-in     (β(t) = ε·cos) →  drive T_lockin steps     →  accumulate H_+

grad = -2 · Re(H_+) / (T_lockin · ε)
```

The lock-in loop is not an extra equilibrium — it is one continued
integration of the same network, with the probe forcing the system
to trace a small orbit around `z*_free`. In analog/neuromorphic
hardware the difference is essential: a free settle is one
fixed-point operation; lock-in needs only that plus a synchronous
demodulator, which is a primitive analog operation.

## 4. One lock-in extraction, step by step

We build a 2-layer phasor chain — the same `_ep_chain` setup the
package tests use — and walk through the four steps of a lock-in
gradient extraction, then verify the result against the finite-
difference (FD) oracle.

In [ ]:
# 2-layer phasor chain: 4 → 8 → 2, real weights, unit-circle activation.
# Matches `_ep_chain` in `test/test_ep.jl`. The width-8 hidden layer is
# narrow enough that the linear-response basin around the free equilibrium
# stays sub-bifurcation at the deep-adiabatic probe defaults (ε=0.01,
# ω_p=0.01) — wider chains need a smaller ε to stay in the regime.
function build_chain(rng; n_in=4, n_hid=8, n_out=2, scale=0.4f0)
    chain = Chain(
        PhasorDense(n_in  => n_hid, normalize_to_unit_circle, use_bias=false),
        PhasorDense(n_hid => n_out, normalize_to_unit_circle, use_bias=false),
    )
    ps, st = Lux.setup(rng, chain)
    ps = (
        layer_1 = merge(ps.layer_1, (weight = scale .* ps.layer_1.weight,)),
        layer_2 = merge(ps.layer_2, (weight = scale .* ps.layer_2.weight,)),
    )
    return chain, ps, st
end

rng   = Xoshiro(42)
chain, ps, st = build_chain(rng)
x     = Phase.(2f0 .* rand(rng, Float32, 4) .- 1f0)
y     = ComplexF32.(exp.(im .* π .* (2f0 .* rand(rng, Float32, 2) .- 1f0)))
cost  = SimilarityCost(y)

println("Network: 4 → 8 → 2 phasor MLP")
println("Input  x  (phases):     ", round.(Float32.(x); digits=3))
println("Target y  (complex):    ", round.(y; digits=3))

In [ ]:
# Step 1: free settle to β = 0 equilibrium and snapshot the DC Hebbians.
ε        = 0.01f0
ω_p      = 0.01f0
dt       = 0.1f0
n_cycles = 8
T_warmup_cycles = 2
T_free   = 400

s_free = phasor_settle(chain, ps, st, x, cost, 0f0; T=T_free, dt=dt)
h_dc   = chain_hebbians(chain, ps, st, x, s_free)

println("Free equilibrium magnitudes:")
println("  layer_1: |z| mean = ", round(mean(abs.(s_free[1])); digits=3))
println("  layer_2: |z| mean = ", round(mean(abs.(s_free[2])); digits=3))

In [ ]:
# Inline lock-in extraction so we can plot H_acc convergence.
# We use phasor_settle with init=states and T=1 to step the recurrence one
# step at a time under the probe β(t). The package's LockinEP runs the same
# loop internally (without recording the trajectory).
_phase_to_complex(x) = ComplexF32.(exp.(im .* π .* Float32.(x)))

period_steps = round(Int, 2π / (ω_p * dt))
T_warmup     = T_warmup_cycles * period_steps
T_lockin     = n_cycles        * period_steps

# Warm-up: drive the probe but don't accumulate (transients die out).
states_w = [copy(s) for s in s_free]
for t in 1:T_warmup
    β_t = ε * cos(ω_p * t * dt)
    global states_w = phasor_settle(chain, ps, st, x, cost, β_t;
                                    T=1, dt=dt, init=states_w)
end

# Lock-in: drive + DC-subtracted demodulated accumulation.
H_acc_traj = ComplexF32[]   # H_acc[1,1] of layer_1 over time, for plotting
H_acc      = zeros(ComplexF32, size(ps.layer_1.weight))
states_l   = [copy(s) for s in states_w]
z_in_layer1 = _phase_to_complex(x)

for t in 1:T_lockin
    β_t = ε * cos(ω_p * t * dt)
    global states_l = phasor_settle(chain, ps, st, x, cost, β_t;
                                    T=1, dt=dt, init=states_l)
    demod = exp(-im * ω_p * t * dt)
    h_W   = states_l[1] * adjoint(z_in_layer1)
    H_acc .+= (h_W .- ComplexF32.(h_dc.layer_1.weight)) .* demod
    push!(H_acc_traj, H_acc[1, 1])
end

# Step 4: convert to gradient — the factor of 2 comes from
# Re(cos(ω_p·t) · e^{-iω_p·t}) averaging to 1/2 over an integer cycle.
g_lockin_layer1 = -2f0 .* real.(H_acc) ./ (T_lockin * ε)

println("Inline lock-in extraction complete.")
println("  T_warmup = ", T_warmup, " steps  (", T_warmup_cycles, " cycles)")
println("  T_lockin = ", T_lockin, " steps  (", n_cycles, " cycles)")
println("  ε = ", ε, "    ω_p = ", ω_p, "    period = ",
        round(2π/ω_p, digits=1), " time units")
println()
println("|H_acc[1,1]| at end of lock-in: ", round(abs(H_acc[1, 1]); digits=4))
println("g_lockin[1,1] (layer_1):         ", round(g_lockin_layer1[1, 1]; digits=4))

In [ ]:
# Running Hebbian-accumulator magnitude over the lock-in loop. As probe
# cycles complete, the demodulated average converges to the true
# linear-response coefficient. Dashed lines mark probe-period boundaries.
plot(1:T_lockin, abs.(H_acc_traj), lw=2,
     xlabel="Lock-in step", ylabel="|H_acc[1,1]| (layer 1)",
     title="Running Hebbian accumulator over the lock-in loop",
     legend=false)
vline!([k * period_steps for k in 1:n_cycles], lw=1, ls=:dash, color=:gray)

In [ ]:
# Verify: package LockinEP gradient vs FD oracle. Reproduces the test logs:
#   LockinEP vs FD on layer_1: cos≈0.999  rel-err≈0.054
#   LockinEP vs FD on layer_2: cos≈0.999  rel-err≈0.037
method = LockinEP(ε=ε, ω_p=ω_p, n_cycles=n_cycles,
                  T_warmup_cycles=T_warmup_cycles, T_free=T_free, dt=dt)
grads_lk, _ = ep_gradient(method, chain, ps, st, x, y)

fd = fd_gradient_phasor(chain, ps, st, x, y; T=200, dt=0.5f0)

for key in (:layer_1, :layer_2)
    g_lk = grads_lk[key].weight
    g_fd = fd[key].weight
    re   = norm(g_lk - g_fd) / norm(g_fd)
    cs   = dot(vec(g_lk), vec(g_fd)) / (norm(g_lk) * norm(g_fd) + 1e-10)
    println("LockinEP vs FD on $key: cos=$(round(cs, digits=4)) ",
            "rel-err=$(round(re, digits=4))")
end

## 5. The three knobs and their failure modes

Three accuracy knobs trade off against each other — the temporal
analog of hEP's $(N, r)$ pair, with one extra dimension:

| knob | controls | too small | too large |
|---|---|---|---|
| $\varepsilon$ | linearity | FD noise floor | $O(\varepsilon^2)$ nonlinearity bias |
| $\omega_p$ | adiabatic following | runtime balloons | non-adiabatic — system can't track |
| $n_{\text{cycles}}$ | demodulator selectivity | harmonic aliasing | diminishing returns, cost |

Below: sweep each knob with the other two pinned at the
deep-adiabatic defaults `(ε=0.01, ω_p=0.01, n_cycles=8, dt=0.1)`.

In [ ]:
# Helper: package LockinEP gradient, return per-layer rel-err vs FD oracle.
function lockin_relerr(chain, ps, st, x, y, fd; kwargs...)
    method = LockinEP(; T_warmup_cycles=2, T_free=400, dt=0.1f0, kwargs...)
    grads, _ = ep_gradient(method, chain, ps, st, x, y)
    re = Dict{Symbol, Float64}()
    for key in (:layer_1, :layer_2)
        g_lk = grads[key].weight
        g_fd = fd[key].weight
        re[key] = norm(g_lk - g_fd) / norm(g_fd)
    end
    return re
end

fd_oracle = fd_gradient_phasor(chain, ps, st, x, y; T=200, dt=0.5f0)

In [ ]:
# Sweep ε: probe-amplitude trade-off.
εs = Float32[1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
re_eps_l1 = Float64[]
re_eps_l2 = Float64[]
for ε_v in εs
    r = lockin_relerr(chain, ps, st, x, y, fd_oracle;
                      ε=ε_v, ω_p=0.01f0, n_cycles=8)
    push!(re_eps_l1, r[:layer_1])
    push!(re_eps_l2, r[:layer_2])
end

plot(εs, re_eps_l1, xscale=:log10, yscale=:log10, lw=2, marker=:circle,
     label="layer_1", xlabel="ε (probe amplitude)",
     ylabel="rel-err vs FD",
     title="Probe-amplitude sweep (ω_p=0.01, n_cycles=8)")
plot!(εs, re_eps_l2, lw=2, marker=:square, label="layer_2")

In [ ]:
# Sweep ω_p: adiabaticity trade-off.
ωs = Float32[1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0]
re_om_l1 = Float64[]
re_om_l2 = Float64[]
for ωp in ωs
    r = lockin_relerr(chain, ps, st, x, y, fd_oracle;
                      ε=0.01f0, ω_p=ωp, n_cycles=8)
    push!(re_om_l1, r[:layer_1])
    push!(re_om_l2, r[:layer_2])
end

plot(ωs, re_om_l1, xscale=:log10, yscale=:log10, lw=2, marker=:circle,
     label="layer_1", xlabel="ω_p (probe angular frequency)",
     ylabel="rel-err vs FD",
     title="Probe-frequency sweep (ε=0.01, n_cycles=8)")
plot!(ωs, re_om_l2, lw=2, marker=:square, label="layer_2")

In [ ]:
# Sweep n_cycles: demodulator selectivity vs cost.
ns = [2, 3, 4, 6, 8, 12, 16, 24]
re_n_l1 = Float64[]
re_n_l2 = Float64[]
for n in ns
    r = lockin_relerr(chain, ps, st, x, y, fd_oracle;
                      ε=0.01f0, ω_p=0.01f0, n_cycles=n)
    push!(re_n_l1, r[:layer_1])
    push!(re_n_l2, r[:layer_2])
end

plot(ns, re_n_l1, yscale=:log10, lw=2, marker=:circle,
     label="layer_1", xlabel="n_cycles",
     ylabel="rel-err vs FD",
     title="Cycles sweep (ε=0.01, ω_p=0.01)")
plot!(ns, re_n_l2, lw=2, marker=:square, label="layer_2")

## 6. The adiabatic basin

A 2-D sweep over $(\varepsilon, \omega_p)$ traces the *adiabatic
basin* — the region where lock-in faithfully recovers FD. Outside
it, either the probe pushes the network nonlinearly (large
$\varepsilon$) or the network cannot track the probe (large
$\omega_p$).

In [ ]:
# 2-D adiabaticity heatmap. 8×8 grid is enough to see the basin.
εs_grid = Float32[3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 5e-1, 7e-1, 1.0]
ωs_grid = Float32[3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 5e-1, 7e-1, 1.0]

Z = fill(NaN, length(ωs_grid), length(εs_grid))
for (i, ωp) in enumerate(ωs_grid), (j, ε_v) in enumerate(εs_grid)
    r = lockin_relerr(chain, ps, st, x, y, fd_oracle;
                      ε=ε_v, ω_p=ωp, n_cycles=4)
    Z[i, j] = log10(max(r[:layer_1], 1e-6))
end

heatmap(εs_grid, ωs_grid, Z, xscale=:log10, yscale=:log10,
        xlabel="ε", ylabel="ω_p",
        title="log₁₀(rel-err) on layer_1 (n_cycles=4)",
        color=:viridis)
scatter!([0.05], [0.05], color=:white, markersize=6, markershape=:star5,
         label="package default (ε=0.05, ω_p=0.05)")
scatter!([0.01], [0.01], color=:red, markersize=6, markershape=:utriangle,
         label="test default (ε=0.01, ω_p=0.01)")

## 7. Canonical 4-corner XOR

We train on the four canonical XOR patterns. A critical detail for
phasor networks: **inputs must be encoded as orthogonal phases, not
antipodal ones.** Phases live in $[-1, 1]$ in units of $\pi$, and the
unit-circle wrap identifies $\text{Phase}(1)$ with $\text{Phase}(-1)$
— both map to the complex number $-1+0i$. So a "binary"
$\pm 1$ encoding collapses every XOR corner to the same complex
input vector and the network cannot distinguish them.

The fix is to use **two orthogonal phases** for the two bit values:

| bit | phase  | angle | complex |
|---|---|---|---|
| 0 | `Phase(0)`   | $0$       | $1+0i$ |
| 1 | `Phase(0.5)` | $\pi/2$   | $0+1i$ |

The four XOR inputs then map to four distinct unit-modulus complex
pairs — `(1,1), (1,i), (i,1), (i,i)` — which the network's first
layer can actually distinguish.

In [ ]:
# Orthogonal-phase XOR inputs. Bit 0 → Phase(0) (1+0i), bit 1 → Phase(0.5) (0+1i).
# The four corners map to four mutually distinguishable complex inputs:
X_xor = Phase.(Float32[0  0    0.5  0.5;
                       0  0.5  0    0.5])
y_xor = Float32[-1, 1, 1, -1]   # XOR truth: same bits → -1, different → +1

println("XOR inputs after angle_to_complex:")
for i in 1:4
    z = angle_to_complex(X_xor[:, i])
    bits = (Float32(X_xor[1, i]) == 0 ? "0" : "1") *
           (Float32(X_xor[2, i]) == 0 ? "0" : "1")
    println("  bits=$bits  label=$(Int(y_xor[i]))  → complex ", round.(z, digits=3))
end

In [ ]:
# Class codewords on the unit circle. Each Y is a random unit-modulus
# vector in C^4 — using random_symbols gives an orthogonal-ish pair
# with cos(Y_POS, Y_NEG) ≈ -0.5, distinct enough that the network
# must produce class-dependent outputs.
const Y_POS = ComplexF32.(angle_to_complex(random_symbols(Xoshiro(11), (4,))))
const Y_NEG = ComplexF32.(angle_to_complex(random_symbols(Xoshiro(13), (4,))))

println("Y_POS = ", round.(Y_POS, digits=3))
println("Y_NEG = ", round.(Y_NEG, digits=3))
println("cos(Y_POS, Y_NEG) = ", round(real(dot(Y_POS, Y_NEG))/4, digits=3))

label_to_y(lab) = lab > 0 ? Y_POS : Y_NEG

# 2 → 16 → 4 phasor chain; n_out=4 to match the codeword dimension.
function build_xor_chain(rng; scale=0.4f0)
    chain = Chain(
        PhasorDense(2  => 16, normalize_to_unit_circle, use_bias=false),
        PhasorDense(16 => 4,  normalize_to_unit_circle, use_bias=false),
    )
    ps, st = Lux.setup(rng, chain)
    ps = (
        layer_1 = merge(ps.layer_1, (weight = scale .* ps.layer_1.weight,)),
        layer_2 = merge(ps.layer_2, (weight = scale .* ps.layer_2.weight,)),
    )
    return chain, ps, st
end

xor_loader = [(X_xor[:, i], label_to_y(y_xor[i])) for i in 1:4]

In [ ]:
# Train with StaticEP. 100 epochs over 4 corners = 400 gradient updates;
# per-example SGD on phasor XOR is slow but reaches a stable
# memorization of (typically) 3 of 4 corners — see the note below.
chain_s, ps_s, st_s = build_xor_chain(Xoshiro(7))
args = PhasorNetworks.Args(lr=0.05, epochs=100)
losses_static, ps_s_trained, _ = ep_train(
    chain_s, ps_s, st_s, xor_loader, args;
    method=StaticEP(β=0.1f0, T_free=100, T_nudge=50, dt=0.5f0))

println("StaticEP:  start=", round(losses_static[1], digits=3),
        "  end=", round(losses_static[end], digits=3))

In [ ]:
# Train with LockinEP from the same init seed for a fair comparison.
chain_l, ps_l, st_l = build_xor_chain(Xoshiro(7))
losses_lockin, ps_l_trained, _ = ep_train(
    chain_l, ps_l, st_l, xor_loader, args;
    method=LockinEP(ε=0.05f0, ω_p=0.05f0, n_cycles=4,
                    T_warmup_cycles=2, T_free=100, dt=0.1f0))

println("LockinEP:  start=", round(losses_lockin[1], digits=3),
        "  end=", round(losses_lockin[end], digits=3))

In [ ]:
# Loss curves averaged per epoch (each epoch sees all 4 corners).
function epoch_mean(losses, n_per_epoch)
    return [mean(losses[((e-1)*n_per_epoch+1):(e*n_per_epoch)])
            for e in 1:(length(losses) ÷ n_per_epoch)]
end

le_s = epoch_mean(losses_static, 4)
le_l = epoch_mean(losses_lockin, 4)

plot(le_s, lw=2, label="StaticEP (β=0.1)", xlabel="Epoch",
     ylabel="mean similarity loss",
     title="StaticEP vs LockinEP on canonical XOR",
     legend=:topright)
plot!(le_l, lw=2, label="LockinEP (ε=0.05, ω_p=0.05)")

In [ ]:
# Per-corner predictions for each trained network. The predicted class
# is whichever codeword has higher cosine similarity to the settled
# output state.
function predict_class(chain, ps, st, x_phase)
    cost = SimilarityCost(Y_POS)   # β=0; cost ignored for settle direction
    s = phasor_settle(chain, ps, st, x_phase, cost, 0f0; T=100, dt=0.5f0)
    z = s[end]
    return real(dot(Y_POS, z)) > real(dot(Y_NEG, z)) ? 1.0f0 : -1.0f0
end

function corner_table(chain, ps_t, st)
    println(rpad("bits", 6), rpad("label", 8), "pred")
    for i in 1:4
        bits = (Float32(X_xor[1, i]) == 0 ? "0" : "1") *
               (Float32(X_xor[2, i]) == 0 ? "0" : "1")
        pred = Int(predict_class(chain, ps_t, st, X_xor[:, i]))
        marker = pred == Int(y_xor[i]) ? "✓" : "✗"
        println(rpad(bits, 6), rpad(Int(y_xor[i]), 8), "$pred  $marker")
    end
    correct = sum(predict_class(chain, ps_t, st, X_xor[:, i]) == y_xor[i] for i in 1:4)
    println("  → $correct/4 correct")
end

println("StaticEP predictions:")
corner_table(chain_s, ps_s_trained, st_s)
println()
println("LockinEP predictions:")
corner_table(chain_l, ps_l_trained, st_l)

### A note on the 3/4 ceiling

Both methods typically reach **3 of 4 corners correct** — neither
hits a perfect 4/4. This is a real limitation of per-example SGD
on this architecture, not a defect specific to lock-in:

- The §5/§6 sweeps already showed that lock-in delivers the same
  gradient quality as static EP (cos ≈ 0.999, rel-err ≈ 5%). Both
  methods see the same gradient signal and reach the same training
  ceiling.
- `ep_train` updates parameters after every single `(x, y)`. With
  only 4 inputs and an XOR labelling, each update from one corner
  partially un-does the previous corner's. The parity structure of
  XOR is exactly what makes this brittle: any consistent
  feature-detector that fits 3 corners necessarily fails the 4th.
- Mini-batched gradients (averaging `ep_gradient` across all 4
  corners before each update) make it *worse* in this case — the
  $+1$ corners' gradient and the $-1$ corners' gradient cancel on
  average, leaving only the codeword-bias term.
- `ep_demo.ipynb`'s tanh network gets >90% on the same XOR with a
  real-valued hidden layer and a richer Hopfield energy. The
  phasor architecture's two extra constraints — real weights and
  per-layer unit-modulus projection — narrow the function class
  enough that XOR is genuinely on the edge of what 2→16→4 can
  represent under per-example phasor SGD.

So the XOR section here is a fair head-to-head of static vs
lock-in *as gradient estimators*, both hitting the same shared
architectural ceiling. Crossing 3/4 → 4/4 with phasor EP is an
open problem and is not what lock-in claims to solve. See
`docs/phasor_ep_design.md` and the memory entry
`hep_gradient_mismatch` for the broader gradient-quality
discussion.

## 8. Key Takeaways

1. **Lock-in EP works where spatial hEP fails.** The unit-modulus
   projector `z/|z|` is non-holomorphic, so the spatial Cauchy
   contour picks up a conjugate sideband that doesn't average out.
   Time-domain probing with a *real* cosine sidesteps the issue
   entirely.

2. **Real probe, not complex.** $\cos(\omega_p t)$ excites both
   sidebands so the lock-in coefficient picks up
   $\partial/\partial\beta + \partial/\partial\bar\beta = d/d\beta_\text{real}$ —
   matching FD on real weights. The complex probe $e^{i\omega_p t}$
   gives only the Wirtinger half and is wrong for non-holomorphic
   recurrences.

3. **One settle, one driven trajectory.** Static EP runs two
   settles per gradient; lock-in runs one settle plus one driven
   integration. The driven integration is not an extra equilibrium —
   it is a small orbit around the free equilibrium that the network
   simply has to track.

4. **Three knobs, three failure modes.** $\varepsilon$ vs noise
   floor / nonlinearity; $\omega_p$ vs runtime / adiabaticity;
   $n_{\text{cycles}}$ vs aliasing / cost. The adiabatic basin (small
   $\varepsilon$, small $\omega_p$, moderate cycles) is wide and easy
   to land in.

5. **Native fit for phasor hardware.** A synchronous demodulator is
   an analog primitive; the `oscillator_bank` infrastructure in
   PhasorNetworks is already an oscillating substrate. Lock-in
   gradients can in principle be extracted by adding a single
   mixer-and-integrator per parameter direction, with no backward
   pass and no extra equilibrium computation.

A natural next step — not shown here — is **frequency-multiplexed
gradient parallelism**: drive several parameter directions with
distinct probe frequencies $\omega_p^{(k)}$ in the same single
settle, and demodulate each one independently. The package does not
ship this yet, but the lock-in mixer machinery here generalizes
directly.

See `docs/phasor_ep_design.md` § "Temporal Cauchy / lock-in detection
for phasor EP" for the full formal write-up.